# Applying Post-Processing Algorithms to the Digital Earth Normalised Radar Backscatter Product for Sentinel-1 (Collection 0)

This notebook demonstrates a number of ways that you can apply post-processing algorithms to Normalised Radar Backscatter data to support analysis.
The key post-processing steps are:
* Masking
* Applying a speckle filter
* Converting to different normalisation conventions (beta0 and sigma0)
* Converting to decibels

## Set-up

For this notebook, you will choose upfront if you want to run the STAC approach, or the Datacube approach.
Uncomment the approach you wish to run, leaving the other approach commented.

In [ ]:
approach = "stac"
# approach = "datacube"

### Import required libraries

This imports python libraries that are used in both approaches, as well as any that are specific to the chosen approach

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from odc.geo import BoundingBox
import xarray as xr

if approach == "stac":
    from pystac_client import Client
    from odc.stac import load, configure_s3_access

if approach == "datacube": 
    import datacube

### Environment set up

In [ ]:
if approach == "stac":
    catalog = "https://explorer.dev.dea.ga.gov.au/stac"
    stac_client = Client.open(catalog)
    configure_s3_access(cloud_defaults=True, aws_unsigned=True)

if approach == "datacube": 
    dc = datacube.Datacube(env="dev", app="S1_Backscatter_Demo")

## Searching and Loading

For this demo, we will load some data in both Australia and Antarctica to demonstrate the impact of different post-processing steps.
To learn more about how to customise loading, check the following notebooks:

* [02_demonstration](02_demonstration.ipynb)
* [03_loading_with_stac](03_loading_with_stac.ipynb)
* [04_loading_with_datacube](04_loading_with_datacube.ipynb)

In the loads, we don't provide an explicit list of measurements to load, ensuring that all measurements are loaded.

### Australian settings

In [ ]:
aus_bbox = BoundingBox(
    left=145.69297,
    bottom=-17.09918,
    right=145.74027,
    top=-17.05394,
    crs="EPSG:4326"
)

aus_start_date = "2025-03-01"
aus_end_date = "2025-03-31"

aus_product_to_load = "ga_s1_nrb_iw_vv_vh_0"
aus_output_crs = "EPSG:3577"

aus_bbox.explore()

### Antarctic AOI

In [ ]:
ant_bbox = BoundingBox(
    left=145.9772,
    bottom=-67.0930,
    right=145.5077,
    top=-67.5106,
    crs="EPSG:4326"
)

ant_start_date = "2024-06-01"
ant_end_date = "2024-08-01"

ant_product_to_load = "ga_s1_nrb_iw_hh_0"
ant_output_crs = "EPSG:3031"

ant_bbox.explore()

### General settings

In [ ]:
output_res = 20 # metres
group_by_method = "solar_day"

### Load data

#### Australia

In [ ]:
if approach == "stac":

    stac_items = stac_client.search(
        collections=[aus_product_to_load],
        datetime=f"{aus_start_date}/{aus_end_date}",
        bbox=aus_bbox.bbox,
    ).item_collection()

    print(f"Found {len(stac_items)} items")

    aus_ds = load(
        items=stac_items,
        intersects=aus_bbox.boundary(),
        crs=aus_output_crs,
        resolution=output_res,
        groupby=group_by_method,
        chunks={},
    )

if approach == "datacube":
    aus_ds = dc.load(
        product=aus_product_to_load,
        lat=(aus_bbox.bottom, aus_bbox.top),
        lon=(aus_bbox.left, aus_bbox.right),
        output_crs=aus_output_crs,
        resolution=(output_res, -output_res),
        time=(aus_start_date, aus_end_date),
        group_by=group_by_method,
        dask_chunks={}
    )

aus_ds

#### Antarctica

In [ ]:
if approach == "stac":

    stac_items = stac_client.search(
        collections=[ant_product_to_load],
        datetime=f"{ant_start_date}/{ant_end_date}",
        bbox=ant_bbox.bbox,
    ).item_collection()

    print(f"Found {len(stac_items)} items")

    ant_ds = load(
        items=stac_items,
        intersects=ant_bbox.boundary(),
        crs=ant_output_crs,
        resolution=output_res,
        groupby=group_by_method,
        chunks={},
    )

if approach == "datacube":
    ant_ds = dc.load(
        product=ant_product_to_load,
        lat=(ant_bbox.bottom, ant_bbox.top),
        lon=(ant_bbox.left, ant_bbox.right),
        output_crs=ant_output_crs,
        resolution=(output_res, -output_res),
        time=(ant_start_date, ant_end_date),
        group_by=group_by_method,
        dask_chunks={}
    )

ant_ds

## Masking

The Digital Earth Normalised Radar Backscatter product for Sentinel-1 comes with a mask that indicates invalid pixels, along with pixels impacted by layover and shadow. 

The masks have the following values:
| Value | Property |
| --- | ----------- |
| 0 | Valid | 
| 1 | Shadow |
| 2 | Layover |
| 3 | Shadow and layover |
| 255 / NaN | Invalid |

The following code displays the masks and shows how to apply them.

In [ ]:
# Set up custom code to visualise the masks
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter

cmap = mcolors.ListedColormap(["#c1deee", "#1f78b4", "#fb9a99", "#7570b3", "#33a02c"])
class_values = [0, 1, 2, 3, 255]
class_bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 255.5]
class_ticks = [0, 1, 2, 3, 126]
class_labels = ["Valid", "Shadow", "Layover", "Shadow and Layover", "Invalid"]
labels_dict = dict(zip(class_ticks, class_labels))

In [ ]:
# Visualise
display_data = aus_ds.isel(time=0)["VV_gamma0"]
display_mask = aus_ds.isel(time=0)["mask"]

# Define figure
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)

# Display raw data
display_data.plot.imshow(ax=ax0, cmap="Greys_r", robust=True)
ax0.set_aspect("equal")
ax0.set_title("Unmasked data")
ax0.axis("off")

# Display masked
display_mask.plot.imshow(
    ax=ax1,
    cmap=cmap,
    norm=mcolors.BoundaryNorm(class_bounds, cmap.N),
    cbar_kwargs={
        'ticks': class_ticks,
        'format': FuncFormatter(lambda v, _: labels_dict.get(int(v), ''))
    }
)

ax1.set_aspect("equal")
ax1.set_title("Mask")
ax1.axis("off")

plt.tight_layout()
plt.show()



### Dilating the mask

You may find that the shadow and layover mask will miss ridge lines in areas with steep terrain.
These often appear as overly bright ridges next to very dark areas.
This can happen as part of the geocoding process if the digital elevation model does not perfectly reflect the true terrain.
As such, we recommend dilating the layover shadow mask to ensure it encompasses ridges and other terrain features.

We have supplied a utility function [masking.py](../de_sar_demo/masking.py) for doing a simple dilation of a binary mask, where values of 1 should be dilated.
As such, we first select non-valid pixels to pass to the `dilate_mask` utility function.
This function has an argument for the disk radius (as a number of pixels) to use for the dilation footprint.
By default, this is set to a value of 3 pixels, but you may need to experiment with this to achieve the best masking for your data.

In [ ]:
from de_sar_demo.masking import dilate_mask

# Create a layer with all non-valid pixels, including both layover and shadow
non_valid_pixels = aus_ds["mask"] != 0

# Create the dilated mask
dilated_mask = dilate_mask(non_valid_pixels, dilation_radius=3)

In [ ]:
# Visualise
display_mask = aus_ds.isel(time=0)["mask"]
display_dilated_mask = dilated_mask.isel(time=0)

# Define figure
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)

# Display the original mask
display_mask.plot.imshow(
    ax=ax0,
    cmap=cmap,
    norm=mcolors.BoundaryNorm(class_bounds, cmap.N),
    cbar_kwargs={
        'ticks': class_ticks,
        'format': FuncFormatter(lambda v, _: labels_dict.get(int(v), ''))
    }
)

ax0.set_aspect("equal")
ax0.set_title("Mask")
ax0.axis("off")

# Display the dilated mask
display_dilated_mask.plot.imshow(ax=ax1)
ax1.set_aspect("equal")
ax1.set_title("Dilated mask")
ax1.axis("off")

plt.tight_layout()
plt.show()


### Applying the mask

To apply the mask, we use xarray's where function, which takes the condition, followed by the values to keep if the condition is true, followed by the values to use if the condition is false. 
The following code creates a new band, `VV_gamma0_masked`, that keeps the original `VV_gamma0` values where the mask is equal to 0, and replaces values with NaN otherwise.

For ease here, we will just apply the mask to the unaltered backscatter data.
The [02_demonstration.ipynb](02_demonstration.ipynb) notebook shows the typical end-to-end workflow of preparing data for analysis.

We display the masked data using a `viridis` colour map, which reveals the locations where the mask has been applied more clearly.

In [ ]:
aus_ds["VV_gamma0_masked"] = xr.where(dilated_mask==0, aus_ds["VV_gamma0"], np.nan)

In [ ]:
# Visualise
display_unmasked = aus_ds.isel(time=0)["VV_gamma0"]
display_masked = aus_ds.isel(time=0)["VV_gamma0_masked"]

# Define figure
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)

# Display the unmasked data
display_unmasked.plot.imshow(ax=ax0, cmap="viridis", robust=True)
ax0.set_aspect("equal")
ax0.set_title("Unmasked data")
ax0.axis("off")

# Display the masked data
display_masked.plot.imshow(ax=ax1, cmap="viridis", robust=True)
ax1.set_aspect("equal")
ax1.set_title("Masked data")
ax1.axis("off")

plt.tight_layout()
plt.show()

## Speckle filtering

### What is speckle?
Speckle is a type of noise in SAR images that comes from observing multiple different scatterers within a single pixel. 
This manifests as differences in phase for light received across the pixel, resulting in interference.
* If the phase from multiple scatterers lines up (constructive interference), then the pixel will appear brighter than expected.
* If the phase from multiple scatterers doesn't line up (destructive interference), then the pixel will appear darker than expected.

Importantly, this means that even though you may illuminate a uniformly covered area (like a field or water body), there will still be constructive and deconstructive interference within any given pixel, resulting in a salt-and-pepper effect where pixels alternate between bright and dark across the area. 
The image below shows an example of a field with the effect of speckle visible across the image.

![The effect of speckle in SAR images.](https://www.researchgate.net/profile/Patrick-Leinenkugel/publication/225006983/figure/fig7/AS:669473518391311@1536626333606/Speckle-in-radar-images.jpg)

*Credit:* Leinenkugel, Patrick. (2010). [The Combined Use of Optical and SAR Data for Large Area Impervious Surface Mapping.](https://www.researchgate.net/publication/225006983_The_Combined_Use_of_Optical_and_SAR_Data_for_Large_Area_Impervious_Surface_Mapping) 


### Speckle filters

Multiple algorithms have been developed to help reduce speckle in SAR imagery, which can assist with image analysis and interpretation. 

#### Lee Filter

A commonly used (and easy to implement) filter is the Lee filter, developed by [Jong-Sen Lee in 1980](https://ieeexplore.ieee.org/document/4766994). 
The filter is applied per pixel, and involves calculating the mean and variance of the pixels in a given window around the pixel. 
We provide an implementation in [speckle_filters.py](../de_sar_demo/speckle_filters.py), along with a function that applies the filter to Xarray DataArrays.
A demonstration is shown below, with a window size of 3 pixels.

For ease here, we will just apply the filter to the unaltered backscatter data. 
The [02_demonstration](02_demonstration.ipynb) notebook shows the typical end-to-end workflow of preparing data for analysis.

In [ ]:
from de_sar_demo.speckle_filters import apply_lee_filter

aus_ds["VV_gamma0_filtered"] = apply_lee_filter(aus_ds["VV_gamma0"], size=3)

In [ ]:
# Visualise
display_unfiltered = aus_ds.isel(time=0)["VV_gamma0"]
display_filtered = aus_ds.isel(time=0)["VV_gamma0_filtered"]

# Define figure
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)

# Display unfiltered
display_unfiltered.plot.imshow(ax=ax0, cmap="Greys_r", robust=True)
ax0.set_aspect("equal")
ax0.set_title("Unfiltered data")
ax0.axis("off")

# Display filtered
display_filtered.plot.imshow(ax=ax1, cmap="Greys_r", robust=True)
ax1.set_aspect("equal")
ax1.set_title("Filtered using Lee filter, window=3")
ax1.axis("off")

plt.tight_layout()
plt.show()

## Converting to different normalisation conventions

Because SAR is a side-looking sensor, the captured backscatter is affected by the viewing angle, as well as the angle of the terrain producing the backscatter. 
Together these factors are captured by a measurement called the incidence angle.

It is common practice to normalise backscatter to get a representation of the backscatter strength per unit area.
However, these is not just one way to do this, but four, and different normalisation conventions may be preferred for different use cases (such as observing hilly terrain vs flat sea-ice). 
As such, normalised radar backscatter can be provided as the following coefficients:

| Coefficient | Normalisation | Common use case |
| ----------- | ------------- | --------------- |
| $\gamma^0$ (gamma0) | Relative to radar line-of-sight | Land, especially over rugged terrain. Minimises the effect of incidence angle. |
| $\sigma^0$ (sigma0) | Relative to the ground | Flat areas, especially sea ice and ocean. Can either account for, or ignore terrain (see below). |
| $\beta^0$ (beta0) | Relative to the radar slant-range | Calibration, or as an intermediate product for other conventions. |

Over land, the most common normalisation convention is $\gamma^0$ (gamma0), which is the primary convention we provide for the product. 
Out of all normalisation conventions, $\gamma^0$ has the lowest sensitivity to incidence angle (the angle between the sensor and the normal vector for the Earth's surface).

Depending on your use case, you may prefer to use a different convention.
The codes below show how to convert the $\gamma^0$ value to a different convention and uses the Antarctic region example as these conversions are most relevant for studying sea ice.

### Converting from $\gamma^0$ to $\beta^0$

While not commonly used in applications, $\beta^0$ can also be a useful intermediate product when converting to $\sigma^0$. 

For this conversion, we supply the `gamma0_to_beta0_ratio` measurement, which provides a per-pixel conversion factor.
The conversion equation is

$$
\begin{aligned} 
\beta^0 &= \gamma^0 \times \text{gamma0\_to\_beta0\_ratio}\\ 
\end{aligned}
$$

In [ ]:
ant_ds["HH_beta0"] = ant_ds["HH_gamma0"] * ant_ds["gamma0_to_beta0_ratio"]

In [ ]:
# Visualise
display_gamma0 = ant_ds.isel(time=0)["HH_gamma0"]
display_beta0 = ant_ds.isel(time=0)["HH_beta0"]

# Define figure
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)

# Display gamma0
display_gamma0.plot.imshow(ax=ax0, cmap="Greys_r", robust=True)
ax0.set_aspect("equal")
ax0.set_title("Gamma0")
ax0.axis("off")

# Display beta0
display_beta0.plot.imshow(ax=ax1, cmap="Greys_r", robust=True)
ax1.set_aspect("equal")
ax1.set_title("Beta0")
ax1.axis("off")

plt.tight_layout()
plt.show()

### Converting from $\gamma^0$ to $\sigma^0$

#### Terrain $\sigma^0$

Terrain sigma0, ${\sigma_t}^0$ is relative to the ground, including terrain.

For this conversion, we supply the `gamma0_to_sigma0_ratio` measurement, which provides a per-pixel conversion factor.
The conversion equation is

$$
\begin{aligned} 
{\sigma_t}^0 &= \gamma^0 \times \text{gamma0\_to\_sigma0\_ratio}
\end{aligned}
$$

In [ ]:
ant_ds["HH_sigma0_terrain"] = ant_ds["HH_gamma0"] * ant_ds["gamma0_to_sigma0_ratio"]

In [ ]:
# Visualise
display_gamma0 = ant_ds.isel(time=0)["HH_gamma0"]
display_sigma0_terrain = ant_ds.isel(time=0)["HH_sigma0_terrain"]

# Define figure
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)

# Display gamma0
display_gamma0.plot.imshow(ax=ax0, cmap="Greys_r", robust=True)
ax0.set_aspect("equal")
ax0.set_title("Gamma0")
ax0.axis("off")

# Display sigma0 terrain
display_sigma0_terrain.plot.imshow(ax=ax1, cmap="Greys_r", robust=True)
ax1.set_aspect("equal")
ax1.set_title("Sigma0 Terrain")
ax1.axis("off")

plt.tight_layout()
plt.show()

#### Ellipsoid $\sigma^0$

Ellipsoid sigma0, ${\sigma_t}^0$ is relative to the ground, without terrain.

For this conversion, there is not a single layer for the conversion factor, but instead involves converting to $\beta_0$ and then multiplying by the sine of the incidence angle.
The equation is as follows:

$$
\begin{aligned} 
{\sigma_e}^0 &= \beta^0 \times \sin ( \text{incidence\_angle} ) \\
&= \gamma^0 \times \text{gamma0\_to\_beta0\_ratio} \times \sin ( \text{incidence\_angle} )
\end{aligned}
$$


Importantly, the `incidence_angle` layer we provide is in units of degrees, so must be converted to radians prior to calculating the sine of the angle.

In [ ]:
ant_ds["HH_sigma0_ellipsoid"] = ant_ds["HH_gamma0"] * ant_ds["gamma0_to_beta0_ratio"] * np.sin(np.radians(ant_ds["incidence_angle"]))

In [ ]:
# Visualise
display_gamma0 = ant_ds.isel(time=0)["HH_gamma0"]
display_sigma0_ellipsoid = ant_ds.isel(time=0)["HH_sigma0_ellipsoid"]

# Define figure
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)

# Display gamma0
display_gamma0.plot.imshow(ax=ax0, cmap="Greys_r", robust=True)
ax0.set_aspect("equal")
ax0.set_title("Gamma0")
ax0.axis("off")

# Display sigma0 ellipsoid
display_sigma0_ellipsoid.plot.imshow(ax=ax1, cmap="Greys_r", robust=True)
ax1.set_aspect("equal")
ax1.set_title("Sigma0 Ellipsoid")
ax1.axis("off")

plt.tight_layout()
plt.show()

## Converting to decibels

The Digital Earth Normalised Radar Backscatter product for Sentinel-1 is provided as linear gamma0. 
For some analyses, it may be beneficial to convert the linear backscatter to decibels (dB).

The conversion equation is 
$${\gamma^0}_{\text{dB}} = 10 \times \log_{10}({\gamma^0}_\text{linear})$$

For ease here, we will just apply the decibel conversion to the unaltered backscatter data. 
The [02_demonstration](02_demonstration.ipynb) notebook shows the typical end-to-end workflow of preparing data for analysis.

In [ ]:
ant_ds["HH_gamma0_db"] = 10*np.log10(ant_ds["HH_gamma0"])

In [ ]:
# Visualise
display_gamma0_linear = ant_ds.isel(time=0)["HH_gamma0"]
display_gamma0_db = ant_ds.isel(time=0)["HH_gamma0_db"]

# Define figure
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(15, 5), sharex=True, sharey=True)

# Display linear gamma0
display_gamma0_linear.plot.imshow(ax=ax0, cmap="Greys_r", robust=True)
ax0.set_aspect("equal")
ax0.set_title("Gamma0 - Linear")
ax0.axis("off")

# Display dB gamma0
display_gamma0_db.plot.imshow(ax=ax1, cmap="Greys_r", robust=True)
ax1.set_aspect("equal")
ax1.set_title("Gamma0 - dB")
ax1.axis("off")

plt.tight_layout()
plt.show()